# EDA – Reviews Sentiment Analysis
Exploratory data analysis for sentiment model training.

In [ ]:
import warnings
import re
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore")
plt.rcParams["figure.figsize"] = (10, 4)
sns.set_theme(style="whitegrid")

RANDOM_STATE = 42
TARGET = "LABEL-rating-category"
ID_LIKE = ["product_id", "brand_id", "product_name", "brand_name", "variation_desc"]

In [ ]:
df = pd.read_csv("data/processed/reviews.csv", low_memory=False)
print(f"Shape: {df.shape}")
df.head(2)

## 1. Feature Overview

In [ ]:
bool_cols = [
    c
    for c in df.columns
    if df[c].dropna().isin([0, 1]).all() and c != TARGET and c not in ID_LIKE
]
num_cols = [
    c
    for c in df.select_dtypes(include="number").columns
    if c not in bool_cols and c != TARGET and c not in ID_LIKE
]
text_cols = [
    c
    for c in df.select_dtypes(include="object").columns
    if df[c].dropna().str.len().mean() > 15 and c not in ID_LIKE
]
cat_cols = [
    c
    for c in df.select_dtypes(include="object").columns
    if c not in text_cols and c not in ID_LIKE
]
vc = df[TARGET].value_counts().sort_index()


print(f"Total columns:   {df.shape[1]}")
print(f"  Text:          {len(text_cols)} -> {text_cols}")
print(f"  Categorical:   {len(cat_cols)} -> {cat_cols}")
print(f"  Boolean (0/1): {len(bool_cols)} -> {bool_cols}")
print(f"  Numerical:     {len(num_cols)} -> {num_cols}")
print(f"Target:      {TARGET} — {len(vc)} classes: {dict(vc)}")

**Dataset:** 1,092,063 rows × 32 columns after preprocessing.

**Features:** 8 numerical, 10 categorical, 6 boolean, 4 text.
Engineered features from Task 1: `review_text_length`, `caps_ratio`, `has_exclamation`.

**Target:** `LABEL-rating-category` — binary (0 = negative [1–3], 1 = positive [4–5]).

## 2. Class Distribution (Target)

In [ ]:
COLORS = {0.0: "#e74c3c", 1.0: "#2ecc71"}
CLASS_LABELS = {0.0: "Negative", 1.0: "Positive"}

counts = df[TARGET].value_counts().sort_index()
pct = (counts / len(df) * 100).round(1)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
counts.plot(
    kind="bar", ax=axes[0], color=[COLORS[k] for k in counts.index], edgecolor="black"
)
axes[0].set_title("Review count by class")
axes[0].set_xticklabels([CLASS_LABELS[k] for k in counts.index], rotation=0)
axes[0].set_ylabel("Count")

pct.plot(
    kind="pie",
    ax=axes[1],
    labels=[f"{CLASS_LABELS[k]} ({v:.1f}%)" for k, v in zip(pct.index, pct)],
    colors=[COLORS[k] for k in pct.index],
    autopct="",
    startangle=90,
)
axes[1].set_title("Class proportions")
axes[1].set_ylabel("")
plt.tight_layout()
plt.show()

**Class imbalance:** 82.1% positive vs 17.9% negative.

## 3. Missing Values

In [ ]:
missing = df.isnull().sum()
missing = missing[missing > 0].sort_values(ascending=False)
missing_pct = (missing / len(df) * 100).round(1)

if len(missing) == 0:
    print("No missing values — preprocessing already handled all NaNs.")
else:
    miss_df = pd.DataFrame({"missing_count": missing, "missing_pct_%": missing_pct})
    display(miss_df)
    missing_pct.plot(kind="barh", figsize=(10, 6), color="steelblue")
    plt.xlabel("% missing values")
    plt.title("Missing values by column (only columns with any missing)")
    plt.tight_layout()
    plt.show()

Only text columns have missing values: `review_title` (28.3%), `highlights` (10.4%), `ingredients` (2.0%).

**Implications:** These fields are optional, users skip them, not data corruption.

**Handling strategies by type:**
- **Text columns** (our case): fill with `''` - model sees "no text", no rows lost. Alternative: drop rows, but 28% loss on `review_title` is too costly.
- **Numerical columns**: median imputation (robust to outliers) or KNN imputation (more accurate, expensive). Drop column if >50% missing.
- **Categorical columns**: fill with `'unknown'` as separate category, or mode imputation. Drop column if near-entirely missing.
- **Target column**: always drop rows — no label = unusable for training.

## 5. Correlations

In [ ]:
corr = df[num_cols].corr()

mask = np.triu(np.ones_like(corr, dtype=bool))

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(
    corr,
    mask=mask,
    annot=True,
    fmt=".2f",
    cmap="coolwarm",
    center=0,
    square=True,
    linewidths=0.5,
    ax=ax,
)
ax.set_title("Correlation matrix (numerical features)")
plt.tight_layout()
plt.show()

**loves_count ↔ reviews** (0.79) — strong positive correlation: popular products get both more likes and more reviews. These features carry redundant information; one could be dropped without losing much signal.

**loves_count ↔ child_count** (0.38), **reviews ↔ child_count** (0.39) — moderate: products with more variants tend to be more popular.

Engineered features (`review_text_length`, `caps_ratio`) show near-zero correlation with all others — they carry independent signal.

In [ ]:
corr_target = df[num_cols + bool_cols].corrwith(df[TARGET]).sort_values()

fig, ax = plt.subplots(figsize=(8, 5))
colors = [COLORS[1.0] if v > 0 else COLORS[0.0] for v in corr_target]
corr_target.plot(kind="barh", color=colors, edgecolor="black", ax=ax)
ax.set_title("Correlation with target (LABEL-rating-category)")
ax.set_xlabel("Pearson r")
ax.axvline(0, color="black", linewidth=0.8)
plt.tight_layout()
plt.show()

print(corr_target.round(3).to_string())

**Strong correlation:** `has_exclamation` (0.26) and `rating` (0.23) correlate most with the target — positive reviews tend to use more exclamation marks and come from higher-rated products.

**Weak but present:** `caps_ratio` (0.03), `new` (0.03) — minimal signal.

**Near-zero:** all remaining features (|r| < 0.02) show virtually no linear correlation with the target — `brand_id`, `loves_count`, `price_usd`, boolean product flags.

**Implication:** Numerical and boolean features alone are weak predictors. The model will likely rely heavily on text (`review_text`).

## 6. Numerical Features – Distributions by Class

In [ ]:
key_num = ["review_text_length", "caps_ratio", "loves_count", "price_usd"]

fig, axes = plt.subplots(1, len(key_num), figsize=(20, 4))
for ax, col in zip(axes, key_num):
    for cls, grp in df.groupby(TARGET):
        ax.hist(
            grp[col].dropna(),
            bins=50,
            alpha=0.5,
            label=CLASS_LABELS[cls],
            color=COLORS[cls],
            density=True,
        )
    ax.set_title(col)
    ax.legend(fontsize=7)
plt.suptitle("Feature distributions by class (numerical)", y=1.02)
plt.tight_layout()
plt.show()

Almost all numerical features show significant overlap between positive and negative classes.

## 7. Categorical Features

In [ ]:
display(
    pd.DataFrame(
        {
            "unique_values": {c: df[c].nunique() for c in cat_cols},
            "missing_pct_%": {
                c: round(df[c].isnull().mean() * 100, 1) for c in cat_cols
            },
        }
    )
)

In [ ]:
for col in ["skin_tone", "skin_type", "primary_category", "secondary_category"]:
    vc = df[col].value_counts()
    pct = (vc / len(df) * 100).round(1)
    print(pd.DataFrame({"count": vc, "pct_%": pct}).head(5).to_string())
    print()

- `primary_category` — only "Skincare" (1 unique value) → zero variance, useless as feature.
- `skin_tone` — dominated by light/fair shades; 15.6% `'unknown'` (users skipped profile).
- `skin_type` — combination skin dominates (49.8%); 10.2% `'unknown'`.
- `secondary_category` — 13 categories, top 3 (Moisturizers, Treatments, Cleansers) cover ~66% of reviews.

## 8. Text Analysis – Top Words by Class

In [ ]:
def top_words(texts: pd.Series, n: int = 20) -> list:
    words = re.findall(r"not_[a-z]+|[a-z]+", " ".join(texts.fillna("").str.lower()))
    return Counter(w for w in words if len(w) > 2).most_common(n)


fig, axes = plt.subplots(1, 2, figsize=(14, 6))
for ax, (cls, label) in zip(axes, CLASS_LABELS.items()):
    subset = df[df[TARGET] == cls]["review_text"]
    words, freqs = zip(*top_words(subset, 20))
    pd.Series(freqs, index=words).plot(kind="barh", ax=ax, color=COLORS[cls])
    ax.set_title(f"Top 20 words – {label}")
    ax.invert_yaxis()
plt.tight_layout()
plt.show()

Top words are dominated by stopwords (`the`, `and`, `this`, `for` etc.) — no sentiment signal visible.
Text cleaning (stopword removal, lemmatization) is required before meaningful analysis.

## 9. Text Characteristics

In [ ]:
display(
    df.groupby(TARGET)["review_text_length"]
    .describe()[["mean", "50%", "max"]]
    .rename(index=CLASS_LABELS)
    .round(0)
)

Negative and positive reviews have nearly identical length distributions (mean ~320, median ~262 chars) — **review length is not a useful predictor of sentiment** in this dataset.

In [ ]:
excl = df.groupby(TARGET)["has_exclamation"].mean().sort_index() * 100
excl.index = [CLASS_LABELS[k] for k in excl.index]

excl.plot(
    kind="bar",
    color=[COLORS[k] for k in sorted(COLORS.keys())],
    edgecolor="black",
    figsize=(7, 4),
)
plt.title('% reviews containing "!" by class')
plt.ylabel("%")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

52% of positive reviews contain "!" vs 18% of negative — exclamation marks are a strong positive sentiment signal, consistent with the 0.26 correlation with target.

## 10. Extra Q1: Does product category affect sentiment?

In [ ]:
cat_sentiment = df.groupby("secondary_category")[TARGET].mean().sort_values()
cat_sentiment.plot(kind="barh", figsize=(10, 5), color="steelblue", edgecolor="black")
plt.axvline(df[TARGET].mean(), color="red", linestyle="--", label="overall mean")
plt.title("Positive rate by secondary_category")
plt.xlabel("% positive reviews")
plt.legend()
plt.tight_layout()
plt.show()

All categories cluster tightly between 75–85% positive — category has minimal impact on sentiment.
"Self Tanners" is the most negative, "Shop by Concern" the most positive, but the
difference is small. `secondary_category` is unlikely to be a useful predictor.

## 11. Extra Q2: Do new products receive better reviews?

In [ ]:
new_sentiment = df.groupby("new")[TARGET].mean()
new_sentiment.index = ["Established", "New product"]

fig, ax = plt.subplots(figsize=(6, 4))
new_sentiment.plot(kind="bar", color=["steelblue", "#9b59b6"], edgecolor="black", ax=ax)
ax.axhline(df[TARGET].mean(), color="red", linestyle="--", label="overall mean")
ax.set_title("Positive rate: New vs Established products")
ax.set_ylabel("% positive reviews")
ax.set_xticklabels(new_sentiment.index, rotation=0)
ax.legend(loc="lower right")
plt.tight_layout()
plt.show()

New products score slightly higher (89%) vs established (82% ≈ overall mean). Likely a novelty/hype
bias — customers who buy new releases tend to be enthusiastic early adopters, or new products are better quality. However, the difference is small, so `new` is unlikely to be a strong predictor on its own.

# EDA Report – Sephora Reviews Sentiment Dataset

## Dataset Overview
1,092,063 rows × 32 columns.
- **Numerical (8):** `loves_count` (product likes), `rating` (mean product rating), `reviews` (review count), `price_usd` (product price), `child_count` (number of variants), `review_text_length` (review length in chars), `caps_ratio` (ratio of capital letters), `brand_id` (brand ID, excluded from model)
- **Categorical (10):** reviewer demographics (`skin_tone`, `eye_color`, `skin_type`, `hair_color`), product info (`size`, `variation_type`, `variation_value`), product taxonomy (`primary_category`, `secondary_category`, `tertiary_category`)
- **Boolean (6):** product flags — `limited_edition`, `new`, `online_only`, `out_of_stock`, `sephora_exclusive`; engineered — `has_exclamation` (review contains "!")
- **Text (4):** `review_text` (main review body), `review_title` (short title), `highlights` (product highlights list), `ingredients` (product ingredients list)

Engineered features (Task 1): `review_text_length`, `caps_ratio`, `has_exclamation`.


## Target
Binary: 0 = negative [1–3], 1 = positive [4–5]. **82.1% positive vs 17.9% negative** — significant
class imbalance. Use `class_weight='balanced'` and report F1-macro, not accuracy.

## Missing Values
Only text columns: `review_title` (28.3%), `highlights` (10.4%), `ingredients` (2.0%) — filled with `''`.
No missing values in numerical or categorical features after preprocessing.

## Correlations
`loves_count ↔ reviews` (0.79) — redundant, consider dropping one.
Engineered features show near-zero inter-correlation — independent signal.

**With target:** `has_exclamation` (0.26) and `rating` (0.23) are strongest predictors.
All other features |r| < 0.03 — numerical/boolean features alone are weak.

## Categorical Features
`primary_category` — single value ("Skincare"), zero variance, excluded from model.
Demographics (`skin_tone`, `skin_type` etc.) — low cardinality (5–15 unique values), straightforward to encode.
`size`, `variation_value` — high cardinality (500+ unique values), encoding will produce many sparse columns.

## Text Analysis
Text is the primary signal. Top words dominated by stopwords — cleaning needed.

Review length is nearly identical across classes (mean ~320 chars) — not a useful predictor.
52% of positive vs 18% of negative reviews contain "!" — strong signal.

## Extra Questions
**Q1 – Category vs sentiment:** All categories 75–85% positive — minimal variation, weak predictor.  
**Q2 – New vs established products:** New products score 89% vs 82% — likely novelty bias, weak signal.

## Recommendations
1. **Text is the primary signal** — focus on cleaning
2. **Class imbalance** — `class_weight='balanced'`, evaluate with F1-macro not accuracy.
3. **Drop `primary_category`** — zero variance.
4. **Consider dropping `reviews` or `loves_count`** — highly correlated, redundant.